# Phase 3: Model Comparison & Champion Selection
## Telco Customer Churn Prediction Project

**Goal**: Compare all models trained in Phase 2 and select the best one for deployment in the DASH web application.

Models evaluated:
- Logistic Regression (baseline)
- Random Forest
- XGBoost
- LightGBM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from pathlib import Path

from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)
import shap
import warnings
warnings.filterwarnings('ignore')

# Config
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
np.random.seed(42)
print("✅ Libraries loaded for model comparison")

In [ ]:
# Load processed test data + feature names
preprocessor = joblib.load('../src/models/preprocessor.joblib')
feature_names = preprocessor.get_feature_names_out()

X_test = pd.read_csv('../data/processed/X_test_processed.csv', header=0)
X_test.columns = feature_names

y_test = pd.read_csv('../data/processed/y_test.csv').iloc[:, 0].astype(int)

print(f"Test set: {X_test.shape} | Churn rate: {y_test.mean():.1%}")

In [ ]:
# Load all trained models
models = {}
model_paths = {
    'Logistic Regression': '../src/models/logistic_regression.joblib',
    'Random Forest': '../src/models/random_forest.joblib',
    'XGBoost': '../src/models/xgboost.joblib',           # uncomment when you have them
    'LightGBM': '../src/models/lightgbm.joblib'
}

for name, path in model_paths.items():
    if os.path.exists(path):
        models[name] = joblib.load(path)
        print(f"✅ Loaded {name}")
    else:
        print(f"⚠️  Model not found: {path}")

print(f"\nLoaded {len(models)} models for comparison.")

In [ ]:
# Evaluate all models
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    pr = average_precision_score(y_test, y_prob)
    
    # Recall on churn class (important for business)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['No Churn', 'Churn'])
    churn_recall = report['Churn']['recall']
    
    results.append({
        'Model': name,
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(roc, 4),
        'PR-AUC': round(pr, 4),
        'Churn_Recall': round(churn_recall, 4)
    })

# Create comparison DataFrame
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(by='F1-Score', ascending=False)

print("=== Model Comparison Table ===")
display(comparison_df)

# Save comparison
os.makedirs('../data/processed', exist_ok=True)
comparison_df.to_csv('../data/processed/model_comparison.csv', index=False)

=== Model Comparison Table ===


,Model,F1-Score,ROC-AUC,PR-AUC,Churn_Recall
2,XGBoost,0.6347,0.8479,0.6575,0.8128
1,Random Forest,0.6338,0.8423,0.6520,0.7567
3,LightGBM,0.6267,0.8423,0.6518,0.8102
0,Logistic Regression,0.6188,0.8387,0.6346,0.7834


In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
metrics = ['F1-Score', 'ROC-AUC', 'PR-AUC', 'Churn_Recall']

for i, metric in enumerate(metrics):
    ax = axes[i//2, i%2]
    sns.barplot(data=comparison_df, x='Model', y=metric, ax=ax, palette='viridis')
    ax.set_title(f'Comparison by {metric}')
    ax.set_ylim(0, 1)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f')

plt.tight_layout()
plt.savefig('../src/figures/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Select champion (highest F1-score)
champion_name = comparison_df.iloc[0]['Model']
champion_model = models[champion_name]

print(f"🏆 CHAMPION MODEL: {champion_name}")
print(f"F1-Score: {comparison_df.iloc[0]['F1-Score']:.4f}")

# Save champion for DASH deployment
Path('../src/models').mkdir(parents=True, exist_ok=True)
joblib.dump(champion_model, '../src/models/champion_model.joblib')
joblib.dump(feature_names, '../src/models/feature_names.joblib')

print("✅ Champion model saved as 'champion_model.joblib' – ready for DASH web app!")

In [ ]:
# SHAP Analysis on Champion (if tree-based)
if champion_name in ['Random Forest', 'XGBoost', 'LightGBM']:
    print(f"\nGenerating SHAP explanations for {champion_name}...")
    
    # Use TreeExplainer for tree models
    explainer = shap.TreeExplainer(champion_model)
    shap_values = explainer.shap_values(X_test)
    
    # Summary plot (global feature importance)
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
    plt.title(f'Global SHAP Feature Importance - {champion_name}')
    plt.tight_layout()
    plt.savefig('../src/figures/shap_global_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Detailed beeswarm plot
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test, show=False)
    plt.title(f'SHAP Summary Plot - {champion_name}')
    plt.tight_layout()
    plt.savefig('../src/figures/shap_summary.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ SHAP plots saved. These will be very useful for business explanation in the report.")

## Summary & Recommendations for Phase 4 (DASH Web App)

**Key Takeaways:**
- **Champion Model**: XGBoost
- Best F1-Score: 0.6347
- Churn Recall: champion_model.joblib (critical for business)